# 05 – Model Training

**Goal:** Train a supervised Random Forest classifier and an unsupervised Isolation Forest anomaly detector.

The two models answer slightly different questions:
- Random Forest learns from the synthetic `threat_label`.
- Isolation Forest learns patterns from the Normal training records and flags unusual records.

These models are educational prototypes and must not be used to make employment or disciplinary decisions.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "dataset"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest

df = pd.read_csv(DATA_DIR / "employee_activity_features.csv")

FEATURES = [
    "login_after_hours",
    "files_downloaded",
    "usb_connected",
    "failed_logins",
    "sensitive_file_access",
    "total_activity_indicators"
]
TARGET = "threat_label"

X = df[FEATURES]
y = df[TARGET].astype(int)
ids = df["employee_id"]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training class counts:")
print(y_train.value_counts())

In [ ]:
# Train supervised Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    max_depth=None
)
rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

In [ ]:
# Train Isolation Forest using only Normal training records
X_train_normal = X_train[y_train == 0]

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)
iso_model.fit(X_train_normal)

print("Isolation Forest training completed.")

In [ ]:
# Save models and feature names
joblib.dump(rf_model, MODEL_DIR / "random_forest_model.joblib")
joblib.dump(iso_model, MODEL_DIR / "isolation_forest_model.joblib")
joblib.dump(FEATURES, MODEL_DIR / "feature_names.joblib")

# Save the held-out test split for the evaluation notebook
test_df = X_test.copy()
test_df.insert(0, "employee_id", id_test.values)
test_df["threat_label"] = y_test.values
test_df.to_csv(MODEL_DIR / "test_data.csv", index=False)

print("Models and test data saved in:", MODEL_DIR)

In [ ]:
# Quick sample predictions from Random Forest
sample_predictions = rf_model.predict(X_test.head(10))
pd.DataFrame({
    "actual_label": y_test.head(10).values,
    "predicted_label": sample_predictions
})

## Training notes

The test split is held out from model fitting. Isolation Forest is fitted only on Normal training examples. `contamination=0.05` is an assumed anomaly proportion for this experiment, not a universal security threshold.